# Pilot Corpus Validation

This notebook validates the original files collected for the first time-aware RAG experiment. It checks source integrity, identifies file formats, attempts text extraction, and compares the 2024 and 2026 versions of the Graduate School Academic Operation Regulations.

Raw source files remain local under `data/raw/` and are not committed to Git. Their official URLs and expected SHA-256 digests are recorded in `data/source_manifest.csv`.

In [1]:
from __future__ import annotations

import hashlib
import re
import zipfile
from pathlib import Path
from xml.etree import ElementTree as ET

import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display
from pypdf import PdfReader


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'source_manifest.csv').exists():
            return candidate
    raise FileNotFoundError('Could not find data/source_manifest.csv')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
MANIFEST_PATH = REPO_ROOT / 'data' / 'source_manifest.csv'
manifest = pd.read_csv(MANIFEST_PATH, keep_default_na=False)

print('Repository root located.')
print(f'Manifest rows: {len(manifest)}')


Repository root located.
Manifest rows: 16


## Extraction helpers

HTML, PDF, DOCX, and HWPX files are parsed with the currently available Python libraries or standard-library modules. Legacy binary HWP files are signature-checked but require a dedicated HWP converter before they can enter the RAG indexing pipeline.

In [2]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def normalize_text(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()


def extract_html(path: Path) -> str:
    soup = BeautifulSoup(path.read_bytes(), 'html.parser')
    for node in soup(['script', 'style', 'noscript']):
        node.decompose()
    return normalize_text(soup.get_text(' ', strip=True))


def extract_pdf(path: Path) -> tuple[str, int]:
    reader = PdfReader(str(path))
    text = ' '.join(page.extract_text() or '' for page in reader.pages)
    return normalize_text(text), len(reader.pages)


def extract_docx(path: Path) -> str:
    with zipfile.ZipFile(path) as archive:
        xml = archive.read('word/document.xml')
    root = ET.fromstring(xml)
    return normalize_text(' '.join(root.itertext()))


def extract_hwpx(path: Path) -> str:
    with zipfile.ZipFile(path) as archive:
        section_names = sorted(
            name for name in archive.namelist()
            if name.lower().startswith('contents/section') and name.lower().endswith('.xml')
        )
        texts = []
        for name in section_names:
            root = ET.fromstring(archive.read(name))
            texts.extend(root.itertext())
    return normalize_text(' '.join(texts))


def signature(path: Path) -> str:
    return path.read_bytes()[:8].hex(' ')


def extract(path: Path) -> tuple[str, str, int | None]:
    suffix = path.suffix.lower()
    if suffix == '.html':
        return extract_html(path), 'extracted', None
    if suffix == '.pdf':
        text, pages = extract_pdf(path)
        return text, 'extracted', pages
    if suffix == '.docx':
        return extract_docx(path), 'extracted', None
    if suffix == '.hwpx':
        return extract_hwpx(path), 'extracted', None
    if suffix == '.hwp':
        return '', 'binary HWP validated; conversion required', None
    return '', f'unsupported extension: {suffix}', None


## File integrity and extraction results

In [3]:
records = []

for row in manifest.itertuples(index=False):
    path = REPO_ROOT / 'data' / row.local_path
    exists = path.exists()
    actual_size = path.stat().st_size if exists else None
    actual_hash = sha256(path) if exists else ''

    text = ''
    extraction_status = 'missing'
    page_count = None
    error = ''
    if exists:
        try:
            text, extraction_status, page_count = extract(path)
        except Exception as exc:
            extraction_status = 'failed'
            error = f'{type(exc).__name__}: {exc}'

    records.append({
        'file': row.local_path,
        'type': row.source_type,
        'version': row.version_status,
        'exists': exists,
        'size_ok': actual_size == int(row.bytes) if exists else False,
        'sha256_ok': actual_hash == row.sha256 if exists else False,
        'signature': signature(path) if exists else '',
        'extraction': extraction_status,
        'pages': page_count,
        'text_chars': len(text),
        'preview': text[:140],
        'error': error,
    })

results = pd.DataFrame(records)
display(results[['file', 'exists', 'size_ok', 'sha256_ok', 'extraction', 'pages', 'text_chars']])


,file,exists,size_ok,sha256_ok,extraction,pages,text_chars
0,raw/law_go_kr/01_university_rules_current_2026...,True,True,True,extracted,NaN,45191
1,raw/law_go_kr/02_graduate_academic_operation_2...,True,True,True,extracted,NaN,11554
2,raw/law_go_kr/03_graduate_academic_operation_c...,True,True,True,extracted,NaN,11936
3,raw/law_go_kr/04_graduate_degree_conferral_cur...,True,True,True,extracted,NaN,4592
4,raw/law_go_kr/05_graduate_curriculum_guideline...,True,True,True,extracted,NaN,991
5,raw/law_go_kr/06_thesis_qualification_exam_gui...,True,True,True,extracted,NaN,1603
6,raw/law_go_kr/07_research_ethics_regulations_2...,True,True,True,extracted,NaN,10296
7,raw/gnu/08_thesis_writing_guidelines_page_2025...,True,True,True,extracted,NaN,2185
8,raw/gnu/08a_thesis_writing_guidelines_ko_en.hwp,True,True,True,binary HWP validated; conversion required,NaN,0
9,raw/gnu/08b_thesis_template_korean.docx,True,True,True,extracted,NaN,1040


In [4]:
summary = pd.DataFrame({
    'metric': [
        'manifest entries',
        'files present',
        'size checks passed',
        'SHA-256 checks passed',
        'text extraction succeeded',
        'legacy HWP conversion required',
        'extraction failures',
    ],
    'value': [
        len(results),
        int(results['exists'].sum()),
        int(results['size_ok'].sum()),
        int(results['sha256_ok'].sum()),
        int((results['extraction'] == 'extracted').sum()),
        int(results['extraction'].str.contains('conversion required').sum()),
        int((results['extraction'] == 'failed').sum()),
    ],
})
display(summary)

assert results['exists'].all(), 'One or more raw files are missing.'
assert results['size_ok'].all(), 'One or more file sizes differ from the manifest.'
assert results['sha256_ok'].all(), 'One or more SHA-256 digests differ from the manifest.'
assert not (results['extraction'] == 'failed').any(), 'One or more extractors failed.'
print('All integrity assertions passed.')


,metric,value
0,manifest entries,16
1,files present,16
2,size checks passed,16
3,SHA-256 checks passed,16
4,text extraction succeeded,15
5,legacy HWP conversion required,1
6,extraction failures,0


All integrity assertions passed.


## Extracted-text preview

Only a short preview is displayed. Full regulation text is intentionally omitted from notebook output.

In [5]:
display(results.loc[results['text_chars'] > 0, ['file', 'text_chars', 'preview']])


,file,text_chars,preview
0,raw/law_go_kr/01_university_rules_current_2026...,45191,"경상국립대학교 학칙 [시행 2026.2.27.] [경상국립대학교학칙 제509호, 2..."
1,raw/law_go_kr/02_graduate_academic_operation_2...,11554,경상국립대학교 대학원 학사운영규정 [시행 2024.1.16.] [경상국립대학교학교규...
2,raw/law_go_kr/03_graduate_academic_operation_c...,11936,경상국립대학교 대학원 학사운영규정 [시행 2026.2.27.] [경상국립대학교학교규...
3,raw/law_go_kr/04_graduate_degree_conferral_cur...,4592,경상국립대학교 대학원 학위수여규정 [시행 2026.2.27.] [경상국립대학교학교규...
4,raw/law_go_kr/05_graduate_curriculum_guideline...,991,경상국립대학교 대학원 교육과정 운영지침 [시행 2024.1.16.] [경상국립대학교...
5,raw/law_go_kr/06_thesis_qualification_exam_gui...,1603,경상국립대학교 일반대학원 학위논문 제출자격시험 시행지침 [시행 2021.11.23....
6,raw/law_go_kr/07_research_ethics_regulations_2...,10296,경상국립대학교 연구윤리 규정 [시행 2025.2.28.] [경상국립대학교학교규정 제...
7,raw/gnu/08_thesis_writing_guidelines_page_2025...,2185,자료실<커뮤니티 | 대학원 메인메뉴 바로가기 본문으로 바로가기 경상국립대학교 바로가...
9,raw/gnu/08b_thesis_template_korean.docx,1040,국문 예시 1) 표지의 양식 석(박)사 학위논문 (16pt ) 지도교수 ○ ○ ○ ...
10,raw/gnu/08c_thesis_template_english.docx,1334,영 문 예시 1) 표지의 양식 A Thesis for the Degree of Ma...


## First version-conflict test

The 2024 and 2026 Graduate School Academic Operation Regulations are compared at the normalized-text level. This is a corpus validation test, not yet an article-level legal diff.

In [6]:
old_path = REPO_ROOT / 'data' / 'raw' / 'law_go_kr' / '02_graduate_academic_operation_2024.html'
current_path = REPO_ROOT / 'data' / 'raw' / 'law_go_kr' / '03_graduate_academic_operation_current_2026.html'
old_text = extract_html(old_path)
current_text = extract_html(current_path)

def unique_articles(text: str) -> set[str]:
    return set(re.findall(r'제\d+조(?:의\d+)?', text))

old_articles = unique_articles(old_text)
current_articles = unique_articles(current_text)
comparison = pd.DataFrame([{
    '2024_text_chars': len(old_text),
    '2026_text_chars': len(current_text),
    '2024_unique_articles': len(old_articles),
    '2026_unique_articles': len(current_articles),
    'articles_only_in_2024': ', '.join(sorted(old_articles - current_articles)),
    'articles_only_in_2026': ', '.join(sorted(current_articles - old_articles)),
    'exact_text_equal': old_text == current_text,
}])
display(comparison)

assert old_text != current_text, 'The historical and current source texts unexpectedly match.'
print('The two official versions are distinct and can be used for the pilot temporal-retrieval test.')


,2024_text_chars,2026_text_chars,2024_unique_articles,2026_unique_articles,articles_only_in_2024,articles_only_in_2026,exact_text_equal
0,11554,11936,50,50,,,False


The two official versions are distinct and can be used for the pilot temporal-retrieval test.


## Validation conclusion

Passing this notebook establishes that the pilot files are present and reproducible from the manifest, and that the supported formats can be converted into text. It does **not** establish that the extracted text is legally complete or that article boundaries are correct. The next notebook should normalize provisions, supplementary provisions, tables, and version metadata before embedding.